In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = Path("../rqs/random_order_strat_8B")
TARGET_K = 8
N_SHUFFLES = 30

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    """Format F1 with two decimals."""
    return f"{x:.2f}"

def fmt_pct(x):
    """Format percentage."""
    if x == np.inf:
        return r"$\infty$"
    return f"{x:.2f}\\%"

def latex_escape(s: str):
    return str(s).replace("_", r"\_")

# ============================================================
# STEP 1: LOAD + COMPUTE METRICS
# ============================================================

summary_rows = []

for repo_dir in BASE_DIR.iterdir():
    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            csv_path = model_dir / "logs" / "order_permutation_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            required_cols = {
                "FewShot_Count",
                "Shuffled_Macro_F1",
                "Original_Macro_F1",
            }

            if not required_cols.issubset(df.columns):
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Shuffled_Macro_F1"] = pd.to_numeric(df["Shuffled_Macro_F1"], errors="coerce")
            df["Original_Macro_F1"] = pd.to_numeric(df["Original_Macro_F1"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count"])
            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            # Combine original + shuffled F1 values
            shuffled = df["Shuffled_Macro_F1"].dropna()
            original = df["Original_Macro_F1"].dropna().unique()

            f1 = pd.concat([
                shuffled,
                pd.Series(original)
            ])

            # Ensure range stays 0–1
            f1 = f1.clip(0, 1)

            min_f1 = f1.min()
            max_f1 = f1.max()
            mean_f1 = f1.mean()
            median_f1 = f1.median()

            delta_f1 = max_f1 - min_f1
            rel_impr = np.inf if min_f1 == 0 else (delta_f1 / min_f1) * 100

            summary_rows.append({
                "Repo": repo,
                "Model": model_dir.name,
                "min_f1": min_f1,
                "max_f1": max_f1,
                "mean_f1": mean_f1,
                "median_f1": median_f1,
                "delta_f1": delta_f1,
                "rel_impr": rel_impr,
            })

summary = pd.DataFrame(summary_rows)
summary = summary.sort_values(["Repo", "Model"]).reset_index(drop=True)

# ============================================================
# STEP 2: BUILD LATEX TABLE
# ============================================================

rows = []

for repo, repo_df in summary.groupby("Repo", sort=False):

    repo_name = REPO_NAME_MAP.get(repo, repo)
    repo_df = repo_df.reset_index(drop=True)

    for i, row in repo_df.iterrows():

        repo_cell = (
            rf"\multirow{{{len(repo_df)}}}{{*}}{{{repo_name}}}"
            if i == 0 else ""
        )

        rows.append(
            f"        {repo_cell} & {latex_escape(row['Model'])} & "
            f"{fmt_f1(row['min_f1'])} & "
            f"{fmt_f1(row['max_f1'])} & "
            f"{fmt_f1(row['mean_f1'])} & "
            f"{fmt_f1(row['median_f1'])} & "
            f"{fmt_f1(row['delta_f1'])} & "
            f"{fmt_pct(row['rel_impr'])} \\\\"
        )

    rows.append("        \\midrule")

rows = rows[:-1]

latex = "\n".join([
    r"\begin{table*}[t]",
    r"    \centering",
    rf"    \caption{{Macro-F1 range across order permutations per repository and model "
    rf"for few-shot count $k = {TARGET_K}$. Entries are computed over the original "
    rf"ordering and {N_SHUFFLES} shuffled orders.}}",
    rf"    \label{{tab:f1-range-order-shuffles-k{TARGET_K}}}",
    r"    \resizebox{\textwidth}{!}{%",
    r"        \begin{tabular}{llrrrrrr}",
    r"            \toprule",
    r"            \textbf{Repository} & \textbf{Model} & "
    r"\textbf{Min F1} & \textbf{Max F1} & "
    r"\textbf{Mean F1} & \textbf{Median F1} & "
    r"\textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\",
    r"            \midrule",
    *rows,
    r"            \bottomrule",
    r"        \end{tabular}",
    r"    }",
    r"\end{table*}",
    "",
])

print(latex)

\begin{table*}[t]
    \centering
    \caption{Macro-F1 range across order permutations per repository and model for few-shot count $k = 8$. Entries are computed over the original ordering and 30 shuffled orders.}
    \label{tab:f1-range-order-shuffles-k8}
    \resizebox{\textwidth}{!}{%
        \begin{tabular}{llrrrrrr}
            \toprule
            \textbf{Repository} & \textbf{Model} & \textbf{Min F1} & \textbf{Max F1} & \textbf{Mean F1} & \textbf{Median F1} & \textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\
            \midrule
        \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.45 & 0.67 & 0.56 & 0.55 & 0.22 & 49.73\% \\
         & Mistral-7B-Instruct-v0.3 & 0.56 & 0.69 & 0.63 & 0.63 & 0.13 & 23.55\% \\
         & Qwen2.5-7B-Instruct & 0.73 & 0.83 & 0.77 & 0.76 & 0.10 & 13.41\% \\
         & gemma-3-12b-it & 0.72 & 0.81 & 0.75 & 0.75 & 0.09 & 12.71\% \\
        \midrule
        \multirow{4}{*}{React} & Llama-3.1-8B-Instruct & 0.24 & 0.63 & 0.47 & 0.48 & 0.39 & 164.30\% \\
 